# 6. The OpenAI-compatible gateway

For plain embeddings you don't need MCP: the gateway (port 8080) implements the OpenAI
embeddings API in front of every service. Any OpenAI client library works, and the `model`
string selects the model as `<service>/<checkpoint>`, e.g. `evo2/7b` or `ntv2/500m`. The
gateway uses the same `MCP_AUTH_TOKEN` as the services, sent as the OpenAI API key.

| Endpoint | |
|---|---|
| `GET /v1/models` | every checkpoint of every running service |
| `POST /v1/embeddings` | one mean-pooled vector per input sequence |
| `GET /version`, `GET /health` | no authentication |

> Run the series in order the first time: notebook 01 explains the setup. Every notebook
> starts by connecting to the services through `tutorial_utils.py` next to it; a service
> that isn't running is skipped.

In [1]:
from collections import Counter

from tutorial_utils import GATEWAY, HBB, HBB_SICKLE, TOKEN, gateway

print("gateway", GATEWAY, "version", gateway("/version")["version"])
models = [m["id"] for m in gateway("/v1/models")["data"]]
print(len(models), "models:", dict(Counter(m.split("/")[0] for m in models)))
print(models[:6], "...")

gateway http://localhost:8080 version 0.9.0
38 models: {'ntv3': 14, 'ntv2': 5, 'dnabert2': 1, 'hyenadna': 5, 'evo2': 1, 'generator': 6, 'genalm': 5, 'grover': 1}
['ntv3/8m-pre', 'ntv3/8m-pre-8kb', 'ntv3/100m-pre', 'ntv3/100m-pre-8kb', 'ntv3/100m-post', 'ntv3/100m-post-131kb'] ...


## Embeddings with the standard library

The request and response follow the OpenAI format: `input` is one sequence or a list,
`data[i].embedding` is the vector, `usage` counts bases.

In [2]:
out = gateway("/v1/embeddings", {"model": "ntv2/500m", "input": [HBB, HBB_SICKLE]})
print(
    out["model"],
    "|",
    len(out["data"]),
    "vectors of",
    len(out["data"][0]["embedding"]),
    "| usage:",
    out["usage"],
)

ntv2/500m | 2 vectors of 1024 | usage: {'prompt_tokens': 180, 'total_tokens': 180}


## With the official OpenAI SDK

```bash
pip install openai
```

The SDK asks for base64-encoded vectors by default and decodes them; the gateway supports
both. Model-specific options that aren't part of the OpenAI API go in `extra_body` and are
passed to the service's `embed_sequence`: `layer` (any layer, e.g. `"20"`), `species` (NTv3
post-trained), `remainder` (GENERator, for lengths that aren't a multiple of 6).

In [3]:
try:
    from openai import OpenAI
except ImportError:
    print("openai is not installed: pip install openai")
else:
    client = OpenAI(base_url=f"{GATEWAY}/v1", api_key=TOKEN or "none")
    resp = client.embeddings.create(model="evo2/7b", input=[HBB, HBB_SICKLE])
    print("evo2/7b:", len(resp.data), "x", len(resp.data[0].embedding))

    resp = client.embeddings.create(model="ntv2/500m", input=HBB, extra_body={"layer": "20"})
    print("ntv2/500m layer 20:", len(resp.data[0].embedding))

    resp = client.embeddings.create(
        model="generator/v2-eukaryote-1.2b", input=HBB + "AG", extra_body={"remainder": "trim_left"}
    )
    print("generator, 92 bp with remainder=trim_left:", len(resp.data[0].embedding))

evo2/7b: 2 x 4096
ntv2/500m layer 20: 1024
generator, 92 bp with remainder=trim_left: 2048


## Errors

Errors use the OpenAI error format, so SDKs raise their usual exceptions: 400 for invalid
input (with the service's message), 404 for an unknown model, 401 for a wrong key, 502 when
a service is down.

In [4]:
import json
import urllib.error

for payload in [
    {"model": "ntv2/500m", "input": "ACGTXACGT"},
    {"model": "no-such-model", "input": "ACGT"},
]:
    try:
        gateway("/v1/embeddings", payload)
    except urllib.error.HTTPError as exc:
        print(exc.code, json.load(exc)["error"]["message"])

400 Error executing tool embed_sequence: Sequence contains invalid character(s) ['X']; only ['A', 'C', 'G', 'N', 'T'] are allowed.
404 Model 'no-such-model' not found. Use '<service>/<checkpoint>' with a service from ['dnabert2', 'evo2', 'genalm', 'generator', 'grover', 'hyenadna', 'ntv2', 'ntv3']; GET /v1/models lists the checkpoints.


## From the command line

```bash
curl -s "$MCP_HOST:8080/v1/embeddings" -H "Authorization: Bearer $MCP_AUTH_TOKEN" \
  -H "Content-Type: application/json" -d '{"model": "hyenadna/medium-160k", "input": "ACGTACGTACGT"}'
```

That's the end of the series. The service READMEs (`services/<name>/README.md`) describe each
model family's checkpoints, limits and licenses in detail.